# Session 3: Stratified K-Fold CV Baseline

This notebook loads the Kaggle credit-card fraud dataset, uses `Class` as the target, and evaluates a logistic regression baseline with stratified 5-fold cross-validation.

Expected data location: put `creditcard.csv` either next to this notebook, in `data/creditcard.csv`, or at `projects/Learning/AI_Engineer/evals/creditcard.csv` when running from the repo root.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
N_SPLITS = 5
THRESHOLD = 0.5
TARGET = "Class"

In [ ]:
candidate_paths = [
    Path("creditcard.csv"),
    Path("data/creditcard.csv"),
    Path("projects/Learning/AI_Engineer/evals/creditcard.csv"),
    Path("projects/Learning/AI_Engineer/evals/data/creditcard.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    expected = "\n".join(f"- {path}" for path in candidate_paths)
    raise FileNotFoundError(
        "Could not find creditcard.csv. Place it in one of these locations "
        "or set data_path manually:\n"
        f"{expected}"
    )

df = pd.read_csv(data_path)
print(f"Loaded {data_path} with shape {df.shape}")
df.head()

In [ ]:
if TARGET not in df.columns:
    raise ValueError(f"Expected target column {TARGET!r}, but columns are: {list(df.columns)}")

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

unique_targets = sorted(y.unique())
if unique_targets != [0, 1]:
    raise ValueError(f"Expected binary target values [0, 1], got {unique_targets}")

numeric_cols = X.select_dtypes(include="number").columns.tolist()
dropped_cols = sorted(set(X.columns) - set(numeric_cols))
if dropped_cols:
    print(f"Dropping non-numeric columns for this baseline: {dropped_cols}")

X = X[numeric_cols]

prevalence = y.mean()
print(f"Rows: {len(df):,}")
print(f"Features used: {len(numeric_cols)}")
print(f"Fraud prevalence overall: {prevalence:.4%}")
y.value_counts().rename(index={0: "not_fraud", 1: "fraud"})

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

model = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1_000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

fold_rows = []

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)

    y_score = model.predict_proba(X_test)[:, 1]
    y_pred = (y_score >= THRESHOLD).astype(int)

    fold_rows.append(
        {
            "fold": fold,
            "train_rows": len(train_idx),
            "test_rows": len(test_idx),
            "train_fraud_prevalence": y_train.mean(),
            "test_fraud_prevalence": y_test.mean(),
            "roc_auc": roc_auc_score(y_test, y_score),
            "pr_auc_avg_precision": average_precision_score(y_test, y_score),
            "f1_at_0_5": f1_score(y_test, y_pred, zero_division=0),
        }
    )

fold_metrics = pd.DataFrame(fold_rows)
fold_metrics

In [ ]:
metric_cols = [
    "train_fraud_prevalence",
    "test_fraud_prevalence",
    "roc_auc",
    "pr_auc_avg_precision",
    "f1_at_0_5",
]

summary = fold_metrics[metric_cols].agg(["mean", "std"]).T
summary

Interpretation notes:

- `test_fraud_prevalence` should be nearly identical across folds. That is the stratification sanity check.
- `ROC-AUC` measures ranking quality across thresholds.
- `PR-AUC / average precision` is usually more informative for rare-positive fraud detection.
- `F1 at 0.5` depends on the fixed threshold. It is useful here as a simple baseline, not as a tuned operating point.

## Deliberately leaky target-mean encoding

This section creates a synthetic high-cardinality categorical column, then computes target means on the full dataset before cross-validation. Because the category is unique per row, the encoded value becomes a disguised copy of `Class`. That is intentionally extreme, but it makes the leakage mechanism easy to see.

In [ ]:
LEAKY_CATEGORY_COL = "synthetic_case_id"
LEAKY_FEATURE_COL = "leaky_target_mean"

df_leaky = df.copy()
df_leaky[LEAKY_CATEGORY_COL] = np.arange(len(df_leaky)).astype(str)

# BAD: this uses every target value, including rows that will later be in test folds.
target_means_before_split = df_leaky.groupby(LEAKY_CATEGORY_COL)[TARGET].mean()
df_leaky[LEAKY_FEATURE_COL] = df_leaky[LEAKY_CATEGORY_COL].map(target_means_before_split)

assert df_leaky[LEAKY_FEATURE_COL].notna().all()
assert np.array_equal(df_leaky[LEAKY_FEATURE_COL].to_numpy(), y.to_numpy())

X_leaky = df_leaky[numeric_cols + [LEAKY_FEATURE_COL]]

print(f"Added {LEAKY_FEATURE_COL!r}. It exactly matches the target, so CV scores should inflate.")
df_leaky[[LEAKY_CATEGORY_COL, TARGET, LEAKY_FEATURE_COL]].head()

In [ ]:
def evaluate_cv(estimator, X_data, y_data, cv_splitter, threshold=THRESHOLD):
    rows = []

    for fold, (train_idx, test_idx) in enumerate(cv_splitter.split(X_data, y_data), start=1):
        X_train, X_test = X_data.iloc[train_idx], X_data.iloc[test_idx]
        y_train, y_test = y_data.iloc[train_idx], y_data.iloc[test_idx]

        fold_model = clone(estimator)
        fold_model.fit(X_train, y_train)

        y_score = fold_model.predict_proba(X_test)[:, 1]
        y_pred = (y_score >= threshold).astype(int)

        rows.append(
            {
                "fold": fold,
                "train_rows": len(train_idx),
                "test_rows": len(test_idx),
                "train_fraud_prevalence": y_train.mean(),
                "test_fraud_prevalence": y_test.mean(),
                "roc_auc": roc_auc_score(y_test, y_score),
                "pr_auc_avg_precision": average_precision_score(y_test, y_score),
                "f1_at_0_5": f1_score(y_test, y_pred, zero_division=0),
            }
        )

    return pd.DataFrame(rows)


leaky_model = clone(model)
leaky_fold_metrics = evaluate_cv(leaky_model, X_leaky, y, cv)
leaky_fold_metrics

In [ ]:
leaky_summary = leaky_fold_metrics[metric_cols].agg(["mean", "std"]).T

pd.concat(
    {
        "baseline_numeric_only": summary,
        "leaky_pre_split_target_mean": leaky_summary,
    },
    names=["experiment", "metric"],
)

## Fix: fit target-mean encoding inside each fold

The corrected version keeps the raw category in `X` and puts the target-mean encoder inside the sklearn pipeline. During cross-validation, each fold learns category means from `y_train` only. Held-out rows are transformed without seeing `y_test`.

In [ ]:
class TargetMeanEncoder(BaseEstimator, TransformerMixin):
    """Encode categorical columns with target means learned during fit()."""

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y):
        X_df = pd.DataFrame(X).copy()
        self.columns_ = list(self.columns)
        y_series = pd.Series(y, index=X_df.index)

        self.global_mean_ = float(y_series.mean())
        self.maps_ = {}
        for col in self.columns_:
            self.maps_[col] = y_series.groupby(X_df[col]).mean()

        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        encoded_cols = []

        for col in self.columns_:
            encoded = X_df[col].map(self.maps_[col]).fillna(self.global_mean_).astype(float)
            encoded_cols.append(encoded.to_numpy())

        return np.column_stack(encoded_cols)


X_fixed = df_leaky[numeric_cols + [LEAKY_CATEGORY_COL]]

fixed_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_cols),
        (
            "target_mean",
            TargetMeanEncoder(columns=[LEAKY_CATEGORY_COL]),
            [LEAKY_CATEGORY_COL],
        ),
    ],
    remainder="drop",
)

fixed_model = Pipeline(
    steps=[
        ("preprocess", fixed_preprocessor),
        (
            "clf",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1_000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

fixed_fold_metrics = evaluate_cv(fixed_model, X_fixed, y, cv)
fixed_fold_metrics

In [ ]:
fixed_summary = fixed_fold_metrics[metric_cols].agg(["mean", "std"]).T

comparison = pd.concat(
    {
        "baseline_numeric_only": summary,
        "leaky_pre_split_target_mean": leaky_summary,
        "fixed_in_fold_target_mean": fixed_summary,
    },
    names=["experiment", "metric"],
)

comparison

In [ ]:
comparison_means = comparison["mean"].unstack("experiment")
display(comparison_means)

assert leaky_summary.loc["roc_auc", "mean"] > summary.loc["roc_auc", "mean"]
assert leaky_summary.loc["pr_auc_avg_precision", "mean"] > summary.loc["pr_auc_avg_precision", "mean"]

print("The leaky pre-split target encoding inflates held-out CV metrics.")
print("The fixed pipeline keeps target encoding inside each fold, so test-fold labels are not used.")

Takeaway:

- Bad pattern: compute target-derived features on the full dataset, then split.
- Correct pattern: put target-derived preprocessing inside the CV pipeline so it is fit only on the training fold.
- For production-grade target encoding, also consider out-of-fold encoding inside the training fold or a library encoder that supports cross-fitting. That avoids memorizing rare categories in the training data itself.